# K00a - GPU smoke test
Clone works, both T4s visible, CUDA matmul runs on each, progress prints, results zip.

In [ ]:
# Reset: always start from a fresh clone of the latest commit (never a stale copy).
import os, shutil, subprocess, sys, time, json
REPO_URL, BRANCH = "https://github.com/saltypal/TrueFace3D.git", "master"
REPO = "/kaggle/working/TrueFace3D" if os.path.isdir("/kaggle") else os.path.abspath("_clone_TrueFace3D")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO], check=True)
COMMIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.chdir(REPO); sys.path.insert(0, REPO)
T0 = time.time()
print(f"[setup] branch {BRANCH} commit {COMMIT} at {time.strftime('%Y-%m-%d %H:%M:%S')}", flush=True)

In [ ]:
# Record what produced these results (lesson 22).
from pathlib import Path
OUT = Path("/kaggle/working/gpu_test"); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "run_info.json").write_text(json.dumps({"commit": COMMIT, "branch": BRANCH,
    "started": time.strftime("%Y-%m-%d %H:%M:%S"),
    "kaggle_kernel": os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "local")}, indent=1))
print("[setup] outputs ->", OUT, flush=True)

In [ ]:
import subprocess; print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout, flush=True)

In [ ]:
import torch, time
res={'torch':torch.__version__,'cuda':torch.version.cuda,'n_gpu':torch.cuda.device_count(),'gpus':[]}
print(res, flush=True)
for i in range(torch.cuda.device_count()):
    d=torch.device(f'cuda:{i}'); a=torch.randn(4096,4096,device=d,dtype=torch.float16); torch.cuda.synchronize(d)
    t=time.time()
    for k in range(50): b=a@a
    torch.cuda.synchronize(d); dt=time.time()-t
    tf=50*2*4096**3/dt/1e12
    g={'gpu':i,'name':torch.cuda.get_device_name(i),'fp16_tflops':round(tf,1),'mem_gb':round(torch.cuda.get_device_properties(i).total_memory/1e9,1)}
    res['gpus'].append(g); print('[gpu]',g, flush=True)
(OUT/'gpu.json').write_text(json.dumps(res,indent=1))
assert res['n_gpu']>=1, 'no GPU'

In [ ]:
# Last cell: one zip of the results only (lessons 18-20).
import fnmatch, zipfile
OUT = Path("/kaggle/working/gpu_test"); ZIP = OUT.with_suffix(".zip")
EXCLUDE = ['*.pth', '*.ckpt', '*.whl', '__pycache__']
n = 0
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and not any(fnmatch.fnmatch(part, pat) for part in p.parts for pat in EXCLUDE):
            zf.write(p, p.relative_to(OUT.parent)); n += 1
print(f"[zip] {n} files -> {ZIP} ({ZIP.stat().st_size / 1e6:.1f} MB); total time {(time.time() - T0) / 60:.1f} min", flush=True)